# AI Cyber Defense Challenge — Nivelul 3 Baseline
## Detectare Open-Set (Known Attacks + Unknown Novel Attacks)

La Nivelul 3, setul de testare conține clase de atacuri noi care NU au existat în setul de antrenare. Atacurile necunoscute trebuie clasificate cu eticheta `Unknown`.

Acest notebook folosește o tehnică bazată pe pragul de încredere (probability thresholding).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score

### 1. Încărcarea datelor (`level3_train.csv` și `level3_test.csv`)

In [ ]:
train_df = pd.read_csv("level3_train.csv")
test_df = pd.read_csv("level3_test.csv")

print(f"Set antrenare: {train_df.shape}")
print(f"Set test: {test_df.shape}")

### 2. Codificarea etichetelor cunoscute

In [ ]:
le = LabelEncoder()
y = le.fit_transform(train_df["attack_cat"])

X = train_df.drop(columns=["attack_cat"])
feature_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
X = X[feature_cols].fillna(0)

test_ids = test_df["id"]
X_test = test_df[feature_cols].fillna(0)

print("Clase cunoscute in antrenare:", list(le.classes_))

### 3. Antrenare model + estimare probabilități pe test

In [ ]:
model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X, y)

# Estimare probabilitati pentru fiecare clasa cunoscuta
probs = model.predict_proba(X_test)
max_probs = np.max(probs, axis=1)
pred_classes = np.argmax(probs, axis=1)

print(f"Probabilitatea medie de incredere: {max_probs.mean():.4f}")

### 4. Aplicarea pragului Open-Set (Thresholding)
Dacă probabilitatea de încredere este mai mică decât un prag, atribuim eticheta `"Unknown"`.
De asemenea, calculăm `unknown_score` continuu pentru metrica AUROC.

In [ ]:
# Folosim un prag bazat pe percentila (top 2% cele mai incerte mostre devin Unknown)
unknown_scores = np.clip(1.0 - max_probs, 0.0, 1.0)
THRESHOLD_SCORE = np.percentile(unknown_scores, 98.0)

known_labels = le.inverse_transform(pred_classes)
final_predictions = np.where(unknown_scores >= THRESHOLD_SCORE, "Unknown", known_labels)

unknown_cnt = int(np.sum(final_predictions == "Unknown"))
print(f"Clasificate ca Unknown: {unknown_cnt} din {len(final_predictions)} ({unknown_cnt/len(final_predictions)*100:.1f}%)")

### 5. Generare submisie `level3_submission.csv`
Formatul cerut conține 3 coloane: `id,attack_cat,unknown_score`.

In [ ]:
submission = pd.DataFrame({
    "id": test_ids,
    "attack_cat": final_predictions,
    "unknown_score": np.round(unknown_scores, 4)
})

submission.to_csv("level3_submission.csv", index=False)
print(f"Fisier generat: level3_submission.csv ({len(submission)} randuri)")
print(submission.head(10))